# ND1 M8 — ROS2 기초 및 로봇 시스템 통합
## 01 빠른 시작 가이드 (Quick Start)

> **목적**: ROS2 설치 전에 M8 모듈의 핵심 개념을 Python 코드로 미리 확인합니다.  
> **환경**: Python 3.10 + numpy + matplotlib (ROS2 불필요)

---
### 이 노트북으로 확인하는 것
1. ROS2 토픽/메시지 구조 시뮬레이션 (CP1)
2. CP1 심화 — LaserScan 이상치 필터링 로직 (가산점)
3. CP2 — 커버리지 계산 로직 시각화
4. CP3 — 목표 좌표 쿼터니언 변환 확인
5. M7 → M8 브릿지 데이터 흐름

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
print('✅ 환경 확인 완료')

## 1. ROS2 토픽 구조 시뮬레이션 (CP1)

```
/cmd_vel  → Twist (linear.x, angular.z)
/odom     → Odometry (position.x, position.y)
/scan     → LaserScan (ranges[360])
/map      → OccupancyGrid (data[], width, height)
```

In [ ]:
class Twist:
    def __init__(self, linear_x=0.0, angular_z=0.0):
        self.linear_x  = linear_x
        self.angular_z = angular_z
    def __repr__(self):
        return f'Twist(v={self.linear_x:.2f} m/s, w={self.angular_z:.2f} rad/s)'

cmd = Twist(linear_x=0.2, angular_z=0.0)
print(f'CP1 cmd_vel 발행: {cmd}')
print(f'  → 0.5초 주기(2 Hz) 타이머로 발행')
print(f'  → ros2 topic hz /cmd_vel 으로 검증')

## 2. CP1 심화 — LaserScan 이상치 필터링

라이다(LaserScan) 데이터에서 무효값을 제거하고 유효 포인트를 RViz2에 시각화합니다.

```
무효값 조건:
  - inf / nan
  - r < range_min  또는  r > range_max

유효 포인트 → 극좌표 → 직교좌표 → MarkerArray (빨간 구)
```

In [ ]:
import math

# LaserScan 시뮬레이션 (360도, 일부 이상치 포함)
np.random.seed(42)
ranges = np.random.uniform(0.3, 3.0, 360).tolist()
# 이상치 주입
for idx in [10, 50, 120, 200, 300]:
    ranges[idx] = float('inf')
ranges[75] = float('nan')
ranges[180] = 0.0  # range_min=0.12 보다 작음

RANGE_MIN = 0.12
RANGE_MAX = 3.5

# 이상치 필터링 (cp1_5_sensor_filter.py 와 동일 로직)
valid_points = []
for i, r in enumerate(ranges):
    if math.isinf(r) or math.isnan(r): continue
    if r < RANGE_MIN or r > RANGE_MAX: continue
    angle = -math.pi + i * (2 * math.pi / 360)
    x = r * math.cos(angle)
    y = r * math.sin(angle)
    valid_points.append((x, y))

print(f'전체 포인트:  {len(ranges)}개')
print(f'유효 포인트:  {len(valid_points)}개')
print(f'제거된 이상치: {len(ranges)-len(valid_points)}개')

# 시각화
xs = [p[0] for p in valid_points]
ys = [p[1] for p in valid_points]
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(xs, ys, s=5, c='red', alpha=0.6, label=f'유효 포인트 ({len(valid_points)}개)')
ax.scatter([0], [0], s=100, c='blue', marker='s', label='로봇 위치')
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('CP1 심화 — LaserScan 이상치 필터링 시뮬레이션')
ax.legend(); ax.grid(True); ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('../results/M8_cp1_shimhwa_sim.png', dpi=100)
plt.show()
print('✅ 저장: results/M8_cp1_shimhwa_sim.png')

## 3. CP2 — 커버리지 계산 시각화

In [ ]:
FREE_THRESHOLD   = 25
LETHAL_THRESHOLD = 65

def calc_coverage(data):
    free     = sum(1 for v in data if 0 <= v <= FREE_THRESHOLD)
    occupied = sum(1 for v in data if v >= LETHAL_THRESHOLD)
    known    = free + occupied
    return free / known * 100.0 if known > 0 else 0.0

steps = list(range(0, 11))
coverage_sim = []
for step in steps:
    n = step * 80
    data_sim = ([-1] * max(0, 1000 - n)
                + [0]   * int(n * 0.75)
                + [100] * int(n * 0.25))
    coverage_sim.append(calc_coverage(data_sim))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, coverage_sim, 'b-o', linewidth=2, label='커버리지 (%)')
ax.axhline(y=70, color='r', linestyle='--', label='목표 70%')
ax.set_xlabel('탐색 스텝'); ax.set_ylabel('커버리지 (%)')
ax.set_title('CP2 — SLAM 커버리지 진행'); ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig('../results/M8_cp2_coverage_sim.png', dpi=100)
plt.show()
print('✅ 저장: results/M8_cp2_coverage_sim.png')

## 4. CP3 — 목표 좌표 + 쿼터니언 변환

In [ ]:
GOALS = [
    {'x':  1.5, 'y':  0.5, 'yaw': 0.0},
    {'x':  2.5, 'y': -1.0, 'yaw': 0.0},
    {'x':  0.5, 'y':  2.0, 'yaw': 0.0},
]

def yaw_to_quat(yaw):
    return {'z': math.sin(yaw/2), 'w': math.cos(yaw/2)}

print('CP3 목표 좌표 + 쿼터니언:')
for i, g in enumerate(GOALS):
    q = yaw_to_quat(g['yaw'])
    print(f'  목표 {i+1}: ({g["x"]:.1f}, {g["y"]:.1f})')
    print(f'    → quat z={q["z"]:.4f}, w={q["w"]:.4f}  |q|²={(q["z"]**2+q["w"]**2):.6f}')

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter([0], [0], s=200, c='green', marker='s', zorder=5, label='출발 (0,0)')
for i, g in enumerate(GOALS):
    ax.scatter(g['x'], g['y'], s=150, c='red', marker='*', zorder=5)
    ax.annotate(f'목표 {i+1}\n({g["x"]},{g["y"]})', (g['x'], g['y']),
                textcoords='offset points', xytext=(8, 8))
ax.set_xlim(-0.5, 3.5); ax.set_ylim(-1.5, 2.5)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('CP3 — 3목표 자율 이동 경로')
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig('../results/M8_cp3_goals.png', dpi=100)
plt.show()
print('✅ 저장: results/M8_cp3_goals.png')

## 5. M7 → M8 데이터 흐름

```
M7 IK 계산 결과 (θ₁, θ₂, θ₃)
    ↓
sensor_msgs/JointState
    name:     ['joint1', 'joint2', 'joint3']
    position: [θ₁, θ₂, θ₃]  ← IK 결과값
    ↓
/joint_states 토픽 발행
    ↓
RViz2 / Gazebo 로봇 팔 시각화
```

In [ ]:
theta_ik = np.array([0.524, -0.785, 1.047])  # 30°, -45°, 60°

print('M7 → M8 브릿지 시뮬레이션')
print(f'  IK 결과 θ: {np.degrees(theta_ik).round(1)} °')
joint_names = ['joint1', 'joint2', 'joint3']
for name, pos in zip(joint_names, theta_ik):
    print(f'    {name}: {pos:.4f} rad ({math.degrees(pos):.1f}°)')
print('  → /joint_states 발행 → RViz2/Gazebo 연동')
print()
print('실행: ros2 run m8_robot m7_to_ros2_bridge')
print('      (ROS2 미설치 시 dry-run 모드로 메시지 구조 출력)')